# UprootRaw ServiceX Tutorial

### Workshop
This is the first tutorial of the [CERN ATLAS ServiceX Summer Workshop 2026](https://indico.cern.ch/event/1699252/)

### Prerequisites
This tutorial assumes that you have completed the [user instalation and setup](https://tryservicex.org/guide/setup.html)

### Introduction
ServiceX offers multiple ways to query for data selection in remote samples. From all the options, UprootRaw covers most of data access needs as it allows.

#### What is UprootRaw?
UprootRaw is a query that leverages a minimal interface for users to provide essential information to be parsed to `uproot` and `awkward` calls executed on the backend. This allows to perform thining (column selection) and skimming (event selection) quickly on remote stored samples for Root TTree and RNTuple files.


This tutorial, based on the [user guide](https://tryservicex.org/guide/) and [previous tutorials](https://github.com/iris-hep/us-atlas-idap-training-2025/blob/main/servicex/servicex_tutorial.ipynb), will go through the main aspects and functionalities of this query type. 

#### Summary
The tutorial will feature 2 main workflows requiring data access:

1) Accessing columns from CP nutples stored on grid to compute a higher-level variable and plot its histogram.
2) Reading xAOD object features through their associated branches from a sample stored in a shared analysis `/eos/` space

These examples contain explanations and bonus documentation.
At the end of the notebook you will find one exercise to build your first UprootRaw queries

#### Make sure your endpoint token `servicex.yaml` is in the running path

In [ ]:
import os
assert os.path.exists('servicex.yaml'), 'servicex.yaml not found in working directory'

## Example 1: Retrieving CP Ntuples sotred in the GRID
### Declare your dataset
The following example uses a Rucio dataset containing CP nutples.

In [ ]:
from servicex import dataset, query, deliver

In [ ]:
my_DS="user.acordeir:michigan-tutorial.displaced-signal.root" 
request_DS=dataset.Rucio(my_DS)

### Building an uproot query. 
This query will define the task the ServiceX workers perform on the selected dataset.

A code is generated on the server using your minimal input for:

1) TTree name (or Rntuple) to open
2) The filtering for branch selection in the tree
3) The event cuts to be applied as masks on the arrays 

Finally, the workers execute the generated code with `uproot` (file reading/writting) and `awkward` (array handling) to thin and skim files before outputing a result that is automatically downloaded on your side. 

In [ ]:
#1 the reco tree from TopCP produced files
tree_name = "reco"

#2 truth signal branches and reco jet variables
branch_selection = [
        "truth_alp_decayVtxX",
        "truth_alp_decayVtxY",
        "truth_alp_pt",
        "truth_alp_eta",
        "jet_EMFrac_NOSYS", 
        "jet_pt_NOSYS" ]

#3 Deciding the cuts in one string with conditions
# Uses awkward array expressions, e.g., count(), num(), all(), any(), max(), abs()...
# A single reco jet, bsm signal pt > 20 GeV & in the inner barrel  
array_mask='(num(jet_pt_NOSYS)<2) & any((truth_alp_pt>20) & (abs(truth_alp_eta)<0.8))'

Select the `UprootRaw` query type from the ServiceX `query` object

In [ ]:
my_uproot_query=query.UprootRaw([{
    'treename': tree_name,
    'filter_name': branch_selection,
    'cut': array_mask
        }])

UprootRaw is one of multiple options, but offers the simplest interface for skimming and thinning ntuples

In [ ]:
print("ServiceX query types:")
print("-"*21)
for plugin in query.plugins:
    print("*", plugin.name, "\n")

### Configure and send the ServiceX request
The query is sent to the server with `deliver()` to be executed. 
To configure it you need to build a simple dictionary with the query, the dataset, and a request name for managing user resquests

In [ ]:
# Define a simple query
config = {
    'Sample': [{
        'Name': 'rucio-displaced-signal',
        'Dataset': request_DS,
        'Query': my_uproot_query
    }]
}

Optional request configuration: 
- `General` block
    - `OutputFormat` for root-file, root-rntuple or parquet
    - `Delivery` to download files to LocalCache or get SignedURLs to the S3 store
- `Sample` sub-block
    - `NFiles` to choose the number of files to transform for each dataset
    - Multiple items can be added. This allows different transforms per `deliver` call.<br>


In [ ]:
#Send the request and wait for results
result = deliver(config)

The deliver return maps the paths of downloaded output files to each request Name:

In [ ]:
print(result)

### Handling the transformed output

 #### The jagged arrays retrieved from the ATLAS events are loaded in `awkward`

In [ ]:
# The analysis_utils package comes with useful tools to handle ServiceX
# to_awk reads the downloaded files and loads/merges them into one awkward array
from servicex_analysis_utils import to_awk
import awkward as ak
import numpy as np

array = to_awk(result)["rucio-displaced-signal"] #to_awk produces arrays for each request in result 
print("Columns retrieved:\n", array.fields)
print(f"\nEvents passing selection in {my_DS}: {len(array)}")

In [ ]:
#Manipulating arrays in awkward
print(f"\nJet momentum average: {ak.mean(array.jet_pt_NOSYS)/1_000:.1e} GeV")

In [ ]:
#Building variables; eg displacement in x-y plane
lxy= (array.truth_alp_decayVtxX**2 + array.truth_alp_decayVtxY**2)**0.5
print(f"\nThe average truth decay vertice displacement is {ak.mean(lxy):.0f} mm")

#### Plot with the `mplhep` library allows you to produce publication-quality plots respecting the ATLAS style

In [ ]:
import matplotlib.pyplot as plt
import mplhep as hep

# ATLAS style
hep.style.use("ATLAS")

# Flatten the awkward array
lxy_flat = ak.to_numpy(ak.flatten(lxy)) #jagged arrays must be flatten

fig, ax = plt.subplots(figsize=(8, 6))

# Histogram
ax.hist(
    lxy_flat,
    bins=np.linspace(0, 4000,50),   # 0 to 4 meters
    histtype="stepfilled",
    edgecolor="black",
    linewidth=1,
    label="Truth signal"
)

# Detector boundaries in Radii
tracker = 1150   # mm
ecal    = 2000   # mm
hcal    = 3800   # mm

# Vertical reference lines
for x, label in [
    (tracker, "End of Tracker"),
    (ecal,    "End of ECAL"),
    (hcal,    "End of HCAL"),
]:
    ax.axvline(
        x,
        color="red",
        linestyle="--",
        linewidth=1.8,
    )

    ax.text(
        x+40,
        0.85,
        label,
        color="red",
        rotation=90,
        va="top",
        ha="left",
        transform=ax.get_xaxis_transform(),
        fontsize=11,
    )

# Labels
ax.set_xlabel(r"$L_{xy}$ [mm]")
ax.set_ylabel("Events")

# ATLAS label
hep.atlas.label(
    "Internal",
    data=False,
    lumi=100, #random for e.g
    com=13.6,
    ax=ax,
)

ax.legend(frameon=False)

plt.tight_layout()

## Example 2: Retrieving columns from DAODs stored on `/eos/`

We just retrieved signal features from an Rucio dataset stored on the grid. However, ServiceX dataset lookup utilities also include `fsspec` and `xRootD` allowing you to query an sample stored on some analysis group space under the `/eos/atlas` path. 


In [ ]:
#Two other useful utilities in the side library
from servicex_analysis_utils import get_structure, ds_type_resolver

In [ ]:
# file stored in a common /eos analysis space
eos_path = "/eos/atlas/atlascerngroupdisk/phys-exotics/ueh/ANA-EXOT-2023-04_A3LP/DAOD_LLP1/mc23_13p6TeV.604615.PhPy8EG_AZNLO_ggH125_mA0p4_Ctau25p0.deriv.DAOD_LLP1.e8601_s4159_r15530_p7073/DAOD_LLP1.47255118._000005.pool.root.1"
daod_dataset = ds_type_resolver(eos_path)

Although UprootRaw cannot extract xAOD object features like in ATLAS software, many features can be read by `uproot` when pointing to the correct branches.

For e.g `xAOD_Jet_V1* j;  j->getAtribute(EMFraction..)` could be accessed via the `AntiKt4EMTopoJetsAuxDyn.EMFrac` branch

Using UprootRaw can still be useful if you need to study specific processes that have not been processed via CP tools such as (TopCP, EasyJet...)

### Using `get_structure` 
This function uses a predefined, specific query to send back the scanned structure of a file from your sample.  
It becomes a very important utility when building queries to verify what branches are available. Additionally, it produces a file metadata output if the `Metadata` tree is in the file.

Example looking for EMFrac branches in the DAOD below:

In [ ]:
print(get_structure(daod_dataset, filter_branch = "EMFrac"))

#### Building the query, configuring the request, and retrieving the result is exactly the same as the first example

In [ ]:
#1 Tree name read by get_structure
tree_daod = "CollectionTree"

#2 get only the EMFrac branch
branch_daod = [
        "AntiKt4EMTopoJetsAuxDyn.EMFrac",
]
#3 no selection here 

In [ ]:
# The cut parameter is optional
daod_uproot_query=query.UprootRaw([{
    'treename': tree_daod,
    'filter_name': branch_daod, 
        }])

In [ ]:
config_daod = {
    'Sample': [{
        'Name': 'DAOD-uproot',
        'Dataset': daod_dataset,
        'Query': daod_uproot_query
    }]
}

In [ ]:
# Send request to ServiceX server
daod_result = deliver(config_daod)

In [ ]:
#Load into arrays
daod_array = to_awk(daod_result)["DAOD-uproot"]

In [ ]:
# ATLAS style
hep.style.use("ATLAS")

# Flatten the awkward array
em_frac= ak.to_numpy(ak.flatten(daod_array["AntiKt4EMTopoJetsAuxDyn.EMFrac"])) #jagged arrays must be flatten

fig, ax = plt.subplots(figsize=(8, 6))

# Histogram
ax.hist(
    em_frac,
    bins=np.linspace(0, 1.1, 20),  
    histtype="stepfilled",
    edgecolor="black",
    linewidth=1,
    label=f"All jets; mean = {np.mean(em_frac):.2f}"
)

# Labels
ax.set_xlabel("Jet EM fraction")
ax.set_ylabel("Events")

# ATLAS label
hep.atlas.label(
    "Internal",
    data=False,
    lumi=100, #random for e.g
    com=13.6,
    ax=ax,
)

ax.legend(frameon=False)

plt.tight_layout()

## Exercise: Plot the di-electron invariant mass
#### You will need to build a ServiceX query that:
1) retrieves the electron's mass and pT branches
2) Selects events with 2 electrons 
3) Selects events where the charge of 2 electrons is opposite 
4) Make two queries, one per dataset, in a single delivery request

Then manipulate your objects with `awkward`, build the invariant mass, and plot it in ATLAS style with `mplhep`

Use `get_structure` to identify the needed branches. (disclaimer: filter_branch does not take wildcards yet and is case-sensitive)

Hints: you can use the `vector` package to build 4-vectors; 2. and 3. can be done in `"cut"`